# ModernBERT Router Smoke — VS Code Colab

이 notebook은 로컬과 Kubeflow가 공유하는 `modernbert-train` CLI를 Colab GPU에서 실행합니다.

1. VS Code 우측 상단 `Select Kernel`을 누릅니다.
2. `Colab`을 선택하고 Google OAuth 로그인을 완료합니다.
3. `New Colab Server`에서 GPU가 있는 machine type을 선택합니다.
4. Explorer에서 `ml/artifacts/modernbert-colab-smoke-input.tar.gz`를 우클릭하고 `Upload to Colab`을 선택합니다.
5. 아래 셀을 순서대로 실행합니다.

업로드 archive는 생성 artifact이며 Git에 포함되지 않습니다. 학습 의미론은 notebook이 아니라 저장소의 Python 모듈에 있습니다.

In [ ]:
import platform
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print({"python": platform.python_version(), "gpu": gpu.stdout.strip() or None})
if gpu.returncode != 0:
    raise RuntimeError("GPU runtime이 아닙니다. Colab의 GPU server를 새로 선택하세요.")

In [ ]:
from pathlib import Path
import shutil
import tarfile

archive = Path("/content/modernbert-colab-smoke-input.tar.gz")
workspace = Path("/content/LLM_Router")
if not archive.is_file():
    raise FileNotFoundError(
        "VS Code Explorer에서 modernbert-colab-smoke-input.tar.gz를 "
        "우클릭해 Upload to Colab을 먼저 실행하세요."
    )
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)
with tarfile.open(archive, "r:gz") as bundle:
    bundle.extractall(workspace, filter="data")
print("Extracted:", workspace)
print("Training data:", (workspace / "ml/artifacts/routerbench-preprocessed-smoke-100/training.parquet").stat().st_size, "bytes")

In [ ]:
import shutil
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--index-url", "https://download.pytorch.org/whl/cu124",
        "--requirement", "/content/LLM_Router/ml/requirements-training-cu124.txt",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--requirement", "/content/LLM_Router/ml/requirements-training.lock",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/LLM_Router/ml"],
    check=True,
)
import torch
import transformers
print({"torch": torch.__version__, "transformers": transformers.__version__, "cuda": torch.cuda.is_available()})

In [ ]:
import os
import subprocess
import sys

root = "/content/LLM_Router"
environment = os.environ.copy()
environment["HF_HOME"] = "/content/huggingface-cache"
executable = shutil.which("modernbert-train")
if executable is None:
    raise RuntimeError("modernbert-train이 없습니다. 바로 앞 설치 셀을 다시 실행하세요.")
command = [
    executable,
    "--config", f"{root}/ml/config/modernbert_smoke.json",
    "--runtime-contract", f"{root}/ml/config/modernbert_runtime_cu124_v1.json",
    "--training-path", f"{root}/ml/artifacts/routerbench-preprocessed-smoke-100/training.parquet",
    "--output-dir", f"{root}/ml/artifacts/modernbert-smoke-colab",
    "--device", "cuda",
]
subprocess.run(command, check=True, env=environment)

In [ ]:
import json
from pathlib import Path

checkpoint = Path("/content/LLM_Router/ml/artifacts/modernbert-smoke-colab")
metrics_path = checkpoint / "metrics.json"
if not metrics_path.is_file():
    raise FileNotFoundError(
        "학습 결과가 없습니다. 바로 앞 modernbert-train 셀을 먼저 실행하고 "
        "해당 셀의 오류를 해결하세요."
    )
metrics = json.loads(metrics_path.read_text())
summary = {
    "runtime_version": metrics["runtime"]["runtime_version"],
    "runtime_contract_passed": metrics["runtime"]["contract_passed"],
    "torch_version": metrics["runtime"]["torch_version"],
    "transformers_version": metrics["runtime"]["transformers_version"],
    "cuda_version": metrics["runtime"]["cuda_version"],
    "device": metrics["model"]["device"],
    "optimizer_steps": metrics["optimization"]["optimizer_steps"],
    "train_loss": metrics["optimization"]["mean_train_loss"],
    "validation_loss": metrics["validation"]["loss"],
    "validation_accuracy": metrics["validation"]["accuracy"],
    "validation_macro_f1": metrics["validation"]["macro_f1"],
    "reload_passed": metrics["checkpoint"]["reload_smoke_passed"],
    "id2label": metrics["checkpoint"]["id2label"],
}
summary

완료 후 Colab Activity Bar의 Contents에서 `/content/LLM_Router/ml/artifacts/modernbert-smoke-colab`을 확인할 수 있습니다. 장기 보존이 필요하면 `Colab: Mount Google Drive to Server...` 명령으로 Drive를 연결한 뒤 checkpoint 폴더를 복사하세요.